# 04 - Resultados, Interpretacion y Recomendacion de Compra

Este notebook cierra el proyecto:

1. Perfil estadistico de cada cluster.
2. Nombre automatico legible por cluster.
3. Etiqueta de prioridad de compra.
4. Proyeccion de unidades para **S2_2026**.
5. Visualizaciones finales.
6. Exportacion del entregable en Excel.

---

In [ ]:
# Permite importar el paquete src desde la carpeta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import utils

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
utils.aplicar_estilo_graficas()

# En notebook queremos ver las figuras en linea, no solo guardarlas en disco
%matplotlib inline

print("Raiz del proyecto:", RAIZ)
print("Archivo fuente existe:", cfg.ARCHIVO_EXCEL.exists())

In [ ]:
from src import clustering as cl
from src import evaluation as ev
from src import feature_engineering as fe

features = fe.cargar_features()
resultado = cl.ejecutar(features)
df = resultado.df
print(f"\nRegistros clasificados: {len(df):,}")

## 1. Perfil de cada cluster

Se combinan dos cosas: la **moda de los atributos** (que tipo de producto
predomina) y el **promedio de las metricas** (como se vende).

In [ ]:
perfiles = ev.perfilar_clusters(df)
columnas = ["Cluster", "n_registros", "moda_Subcategoria", "moda_Marca", "moda_Canal",
            "promedio_mensual_prom", "cobertura_meses_prom", "tendencia_prom",
            "estacionalidad_prom", "variabilidad_prom"]
perfiles[columnas]

## 2. Nombre automatico del cluster

El nombre se **deriva de los datos**, no se escribe a mano. Se arma con cuatro rasgos:

| Rasgo | Como se decide |
|---|---|
| **Nivel de rotacion** | terciles del promedio mensual *entre los clusters* |
| **Dinamica** | ingreso reciente / en crecimiento / en declive, segun tendencia y cobertura |
| **Subcategoria** | moda del cluster |
| **Cierre** | estacionalidad si es marcada; si no, el canal dominante |

**Por que cortes relativos y no umbrales fijos.** Si "alta rotacion" fuera
"mas de 100 unidades/mes", el nombre quedaria obsoleto en cuanto cambiara la
escala del negocio o se agregara un canal. Con terciles, el nombre sigue siendo
correcto porque describe la posicion *dentro de este portafolio*.

In [ ]:
df, perfiles = ev.asignar_nombres(df, perfiles)

print("Nombres asignados:\n")
for _, fila in perfiles.iterrows():
    print(f"  Cluster {int(fila['Cluster']):>2}: {fila['Nombre_Cluster']}")
    print(f"            {int(fila['n_registros'])} registros | "
          f"prom {fila['promedio_mensual_prom']:.1f} und/mes | "
          f"cobertura {fila['cobertura_meses_prom']:.1f}% | "
          f"tendencia {fila['tendencia_prom']:+.1f} und/mes")

### Que significa cada grupo para la compra

- **Alta Rotacion**: el nucleo del negocio. Nunca deben faltar; un quiebre aqui
  cuesta venta real.
- **Rotacion Media**: compra ajustada al patron semestral del grupo.
- **Bajo Movimiento**: revisar caso por caso antes de recomprar.
- **En Declive**: reducir cantidad o evaluar salida del portafolio.
- **Ingreso Reciente**: poca historia propia; la referencia la da el cluster.
- **Producto Nuevo Sin Historial**: el modelo no proyecta, decide el area comercial.

## 3. Prioridad de compra

Reglas, en orden de evaluacion:

| Prioridad | Condicion |
|---|---|
| **Sin Movimiento** | nunca vendio |
| **Alta** | cobertura > 70% **Y** tendencia positiva |
| **Baja** | cobertura < 40% **O** caida fuerte de tendencia |
| **Media** | el resto |

**El orden importa.** "Alta" se evalua despues de "Baja" y la sobrescribe, pero
la condicion de Alta exige tendencia positiva. Asi, un producto que esta siempre
disponible pero en caida fuerte **no** sale como prioridad Alta solo por su
cobertura: cae en Baja, que es la senal correcta para el comprador.

In [ ]:
df = ev.asignar_prioridad(df)

tabla = pd.crosstab(df["Nombre_Cluster"], df["Prioridad_Compra"], margins=True, margins_name="Total")
tabla

In [ ]:
# Cruce prioridad x canal: cuanto pesa cada grupo en cada canal
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

cruce = pd.crosstab(df["Canal"], df["Prioridad_Compra"], normalize="index") * 100
cruce.plot(kind="bar", stacked=True, ax=axes[0], colormap="RdYlGn_r", width=0.7)
axes[0].set_title("Composicion de prioridad por canal")
axes[0].set_ylabel("% de registros")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Prioridad", bbox_to_anchor=(1.0, 1), loc="upper left")

conteo = df["Prioridad_Compra"].value_counts()
axes[1].pie(conteo.values, labels=conteo.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Distribucion global de prioridad")
plt.show()

## 4. Proyeccion para S2_2026

Aqui es donde el clustering paga. La logica tiene dos piezas.

### 4.1 Factor de paso S1 a S2, por cluster

Para cada cluster: cuantas unidades vendio en S2_2025 por cada unidad vendida en
S1_2025. Es el patron semestral tipico del grupo.

**Por que a nivel de cluster y no de producto.** Un SKU individual puede tener un
semestre raro por un quiebre de stock o una promocion puntual. El grupo promedia
ese ruido. **Este es el aporte concreto del modelo a la planeacion.**

El factor se acota entre 0,5x y 2,0x: un factor mas extremo casi siempre viene de
un cluster con base pequena y llevaria a sobre o sub-comprar de forma grave.

In [ ]:
factores = ev.calcular_factor_cluster(df)

### 4.2 Jerarquia de metodos

Se aplica el primero que corresponda a cada producto:

| # | Metodo | Cuando aplica | Por que |
|---|---|---|---|
| 1 | `S1_2026 x factor cluster` | vendio en el semestre mas reciente | parte de un dato observado y actual |
| 2 | `Promedio S2 propio` | sin S1_2026, pero con S2 anteriores | mismo semestre, mismo SKU |
| 3 | `Promedio S2 del cluster` | con venta, sin ningun S2 propio | el grupo aporta la referencia que el producto no tiene |
| 4 | `Sin historial` | nunca vendio | lo define el area comercial |

El resultado se **redondea hacia arriba**: en una compra semestral un excedente
marginal cuesta menos que un quiebre de stock, que se lleva la venta completa.

In [ ]:
df = ev.proyectar_semestre_objetivo(df)

columnas = ["Descripcion", "Canal", "Nombre_Cluster", "unidades_S2_2025",
            "unidades_S1_2026", "factor_cluster_S2", "unidades_sugeridas_S2_2026",
            "metodo_proyeccion"]
df.nlargest(12, "unidades_sugeridas_S2_2026")[columnas]

In [ ]:
# Validacion de escala: la proyeccion es coherente con la historia?
print("Unidades por semestre (historico) y proyeccion:\n")
for semestre in cfg.SEMESTRES:
    print(f"  {semestre}: {utils.formatear_miles(df[f'unidades_{semestre}'].sum()):>12}")
print(f"  {cfg.SEMESTRE_OBJETIVO}: {utils.formatear_miles(df['unidades_sugeridas_S2_2026'].sum()):>12}  <- proyeccion")

crecimiento = (df["unidades_sugeridas_S2_2026"].sum() / df["unidades_S1_2026"].sum() - 1) * 100
print(f"\nVariacion proyectada frente a S1_2026: {crecimiento:+.1f}%")
print("Referencia historica de la misma transicion:")
for anio in [2024, 2025]:
    s1 = df[f"unidades_S1_{anio}"].sum()
    s2 = df[f"unidades_S2_{anio}"].sum()
    print(f"  S1_{anio} -> S2_{anio}: {(s2 / s1 - 1) * 100:+.1f}%")

### Lectura

La variacion proyectada debe quedar en el mismo orden de magnitud que las
transiciones S1 a S2 historicas. Si se saliera mucho de ese rango habria que
revisar el factor de cluster antes de usar la cifra.

In [ ]:
# Resumen por canal: la cifra que va a la mesa de compra
resumen_canal = df.groupby("Canal").agg(
    registros=("Material", "count"),
    prioridad_alta=("Prioridad_Compra", lambda s: (s == "Alta").sum()),
    venta_S1_2026=("unidades_S1_2026", "sum"),
    sugerido_S2_2026=("unidades_sugeridas_S2_2026", "sum"),
    ingreso_proyectado=("ingreso_proyectado_S2_2026", "sum"),
)
resumen_canal["variacion_vs_S1_2026_%"] = (
    (resumen_canal["sugerido_S2_2026"] / resumen_canal["venta_S1_2026"] - 1) * 100
).round(1)
resumen_canal

## 5. Visualizaciones finales

In [ ]:
df, perfiles = ev.asignar_nombres(df, ev.perfilar_clusters(df))

fig = ev.graficar_pca(df)
plt.show()

In [ ]:
fig = ev.graficar_tsne(df, resultado.matriz_pca)
plt.show()

### PCA frente a t-SNE

El **PCA** es una proyeccion lineal: las distancias globales son fieles, asi que
sirve para ver si un cluster esta cerca o lejos de otro.

El **t-SNE** es no lineal y preserva la vecindad local: separa visualmente grupos
que el PCA superpone, pero **las distancias entre grupos no son interpretables**.
Sirve para confirmar que los grupos existen, no para medir cuan diferentes son.

In [ ]:
fig = ev.graficar_perfil_clusters(df)
plt.show()

In [ ]:
fig = ev.graficar_metricas_por_cluster(perfiles)
plt.show()

In [ ]:
# Patron semestral de cada cluster: en cual compra pedir cada grupo
cols_sem = [f"unidades_{s}" for s in cfg.SEMESTRES]
patron = df.groupby("Nombre_Cluster")[cols_sem].mean()
patron.columns = [c.replace("unidades_", "") for c in patron.columns]

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(patron, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax,
            cbar_kws={"label": "Promedio de unidades por registro"})
ax.set_title("Patron de venta semestral promedio por cluster")
ax.set_xlabel("Semestre")
ax.set_ylabel("")
plt.show()

## 6. Tabla resumen para el negocio

In [ ]:
resumen = ev.tabla_resumen(perfiles)
resumen

## 7. Exportacion del entregable

El Excel final tiene 4 hojas:

| Hoja | Contenido |
|---|---|
| `Productos_Clasificados` | dataset completo con Cluster, Nombre_Cluster y Prioridad_Compra |
| `Perfiles_Cluster` | resumen estadistico por cluster |
| `Ventas_por_Semestre` | tabla producto x semestre con totales |
| `Recomendacion_Compra` | proyeccion S2_2026 por producto y canal, con el metodo usado |

In [ ]:
df_final, perfiles_final = ev.ejecutar(resultado)
ruta = utils.exportar_excel_final(df_final, perfiles_final)
print(f"\nEntregable generado: {ruta}")

In [ ]:
# Verificacion de lo exportado
excel = pd.ExcelFile(cfg.ARCHIVO_EXCEL_FINAL)
print("Hojas del entregable:\n")
for hoja in excel.sheet_names:
    tabla = pd.read_excel(cfg.ARCHIVO_EXCEL_FINAL, sheet_name=hoja)
    print(f"  {hoja:<25} {len(tabla):>6} filas x {tabla.shape[1]:>3} columnas")

---

## Cierre del proyecto

**Que quedo construido**

- 1.815 registros producto x canal clasificados, sin excepciones.
- 6 clusters de comportamiento con nombre derivado de los datos, mas un grupo
  reservado para productos sin historial.
- Una etiqueta de prioridad de compra por registro.
- Una cantidad sugerida para S2_2026 por producto y canal, con el metodo de
  calculo indicado en cada fila (para que el comprador sepa cuanta confianza
  darle).
- Dashboard interactivo: `streamlit run dashboard/app.py`

**Limitaciones que conviene tener presentes al usar el resultado**

1. **El Silhouette es moderado (alrededor de 0,20).** Los productos forman un
   continuo mas que grupos nitidamente separados. Los clusters son utiles como
   segmentos de politica de compra, no como categorias naturales indiscutibles.

2. **El modelo solo ve historia de venta.** No conoce quiebres de stock,
   promociones, cambios de exhibicion ni acciones de la competencia. Un producto
   que vendio poco por falta de inventario se ve igual que uno que vendio poco por
   falta de demanda. **Esta es la limitacion mas importante del modelo.**

3. **Los productos sin historial no se proyectan.** Son cerca del 21% de los
   registros y su cantidad la debe definir el area comercial.

4. **El factor de cluster esta acotado a [0,5x , 2,0x].** Los clusters cuyo
   factor real toca el techo quedan con una proyeccion conservadora a proposito.

5. **Solo hay 2 anios completos** para estimar estacionalidad. Con un tercer ano
   la estimacion mejorara.

**Recomendaciones para la siguiente iteracion**

- Incorporar dias sin stock para distinguir demanda real de venta perdida.
- Marcar los meses con promocion, para no leer un pico promocional como tendencia.
- Volver a correr el pipeline al cierre de cada semestre: el factor de paso se
  recalibra automaticamente con el dato nuevo.